# Laboratorio 2: Codificación de fuente (Parte 1)
**Curso:** Teoría de la Información Transmisión de Datos

En este laboratorio estudiaremos los conceptos fundamentales de la codificación de la fuente, específicamente la codificación de símbolos, un pilar de la teoría de la información formulada por Claude Shannon.

Se abordarán los siguientes temas:
* Entropía de una fuente discreta.
* Código de Huffman.

## Configuración Inicial

In [28]:
import heapq
from collections import defaultdict
import math

## Ejemplo: Codificación de Huffman para una fuente discreta de 5 símbolos
En este ejercicio se ilustra cómo aplicar la codificación de Huffman a una fuente discreta con cinco símbolos $ \{S1,S2,S3,S4,S5 \} $ y probabilidades conocidas $ [0.25,0.25,0.20,0.15,0.15] $.

El procedimiento se desarrolla en varias etapas:

* Definición de la fuente: Se listan los símbolos y sus probabilidades, representando la distribución de ocurrencia de eventos en un sistema.

* Cálculo de la entropía: Se obtiene el límite teórico de compresión sin pérdida. La entropía indica la mínima cantidad promedio de bits necesaria para representar cada símbolo.

* Construcción del código de Huffman: Mediante un algoritmo bottom-up, se combinan los símbolos menos probables en un árbol binario, asignando códigos más cortos a los símbolos más frecuentes y más largos a los menos frecuentes. Esto garantiza un código prefijo óptimo.

* Evaluación del rendimiento: Se calcula la longitud promedio del código y se compara con la entropía para medir la eficiencia de la codificación
$ η=H(X)/L(C,X) $.

In [29]:
# =======================================
# 1. Definición de la fuente
# =======================================
symbols = ['S1', 'S2', 'S3', 'S4', 'S5']
probs =   [0.25, 0.25, 0.20, 0.15, 0.15]

# Mostrar distribución
for s, p in zip(symbols, probs):
    print(f"{s}: {p:.2f}")

# =======================================
# 2. Cálculo la entropía de la fuente
# =======================================
H = -sum(p * math.log2(p) for p in probs)
print(f"Entropía H(X) = {H:.3f} bits/símbolo")

# =======================================
# 3. Construcción del código de Huffman
# =======================================
def huffman(symbols, probs):
    heap = [[p, [s, ""]] for s, p in zip(symbols, probs)]
    heapq.heapify(heap)
    while len(heap) > 1:
        lo = heapq.heappop(heap)
        hi = heapq.heappop(heap)
        for pair in lo[1:]:
            pair[1] = '0' + pair[1]
        for pair in hi[1:]:
            pair[1] = '1' + pair[1]
        heapq.heappush(heap, [lo[0] + hi[0]] + lo[1:] + hi[1:])
    return sorted(heapq.heappop(heap)[1:], key=lambda p: (len(p[1]), p))

codes = huffman(symbols, probs)

print("Códigos de Huffman:")
for sym, code in codes:
    print(f"{sym}: {code}")

# =======================================
# 5. Cálculo la longitud promedio y eficiencia
# =======================================
L = sum(p * len(code) for (sym, code), p in zip(codes, probs))
eff = H / L

print(f"Longitud promedio L(C,X) = {L:.3f} bits/símbolo")
print(f"Eficiencia η = {eff*100:.2f}%")

S1: 0.25
S2: 0.25
S3: 0.20
S4: 0.15
S5: 0.15
Entropía H(X) = 2.285 bits/símbolo
Códigos de Huffman:
S1: 01
S2: 10
S3: 00
S4: 110
S5: 111
Longitud promedio L(C,X) = 2.300 bits/símbolo
Eficiencia η = 99.37%


## Ejercicio: Codificación de texto con Huffman

Implemente un flujo completo de compresión sin pérdida basado en codificación de Huffman sobre archivos de texto, evaluando su eficiencia y verificando la decodificación exacta.

**Dataset**

Usar el archivo de texto plano "[Quijote.txt](https://ev1.utec.edu.uy/moodle/pluginfile.php/1303581/mod_assign/introattachment/0/Quijote.txt?forcedownload=1)"

**Tareas**
1) Lectura del fichero:
Cargar el archivo ("[Quijote.txt](https://ev1.utec.edu.uy/moodle/pluginfile.php/1303581/mod_assign/introattachment/0/Quijote.txt?forcedownload=1)") preservando codificación UTF-8.

2) Modelado de la fuente:
Considerar cada carácter como símbolo. Calcular frecuencia absoluta y frecuencia relativa.
Reportar: tamaño en bytes (carácteres) del archivo, número de símbolos distintos, entropía

3) Construcción del código de Huffman:
Generar un código prefijo óptimo con el alfabeto observado.

4) Codificación eficiente a fichero:
Codificar archivo a un fichero binario .huff que contenga el bitstream concatenado de los códigos.

5) Decodificación y verificación:
Decodificar el fichero .huff y verificar su integridad y similitud al original

6) Comparación de tamaños:
Comparar tamaño (bytes) del archivo original y el codificado.

**Requisitos técnicos**

* Tratar correctamente caracteres no ASCII (UTF-8).
* Gestionar padding en el último byte y registrarlo para decodificar sin ambigüedad.
* Validar el propiedad prefijo y la única decodificación.

**Entregables**
* Script(s)/notebook con:
Construcción de Huffman, codificación y decodificación.
Tablas/resúmenes.
Breve discusión: metodología, decisiones de formato y resultados
* Ficheros .huff.


In [30]:
################################################################################################
# Escriba el código del ejercio en esta celda
################################################################################################
import heapq
from collections import defaultdict
import math
import urllib.request
import os
from decimal import Decimal, getcontext
from tabulate import tabulate

# Establece la precisión deseada (50 dígitos)
getcontext().prec = 50

In [31]:
################################################################################################
# 1. Leer archivo desde una URL
################################################################################################

# Url del archivo Quijote.txt compartido desde mi Drive de Utec en una parpeta publica
# Para que no haya que loguearse par alograr que el Notebook de >Colab funcione
################################################################################################
url = "https://drive.google.com/uc?export=download&id=1IBWsnFZ3OsFrFosYOQFJ8BRulLEBs3Cq"


# Descargar y decodificar el archivo (en UTF-8 segun la letra del ejercicio)
################################################################################################
respuesta = urllib.request.urlopen(url)
texto = respuesta.read().decode('utf-8')

In [32]:
################################################################################################
# 2. Construcción de la tabla de ocurrencias
################################################################################################
# Inicializar el diccionario con un valor predeterminado de [0, 0, '']
# El diccionario cuya clave sera cada simbolo encontrado, tendra dentro un vector que
# almacenara [ocurrencias, probabilidad, 'código_huffman']
frecuencia = defaultdict(lambda: [0, Decimal(0), '']) ## diccionario que inicializa automáticamente valores inexistentes a 0
                                                      ## No necesita verificar la existencia de la clave

for char in texto:                                    # para cada carecter del texto le suma 1 a la cantidad de ocurrencias
    frecuencia[char][0] += 1

total_chars = len(texto)
print(f"Cantidad de caracteres del texto a analizar : {total_chars}")
simbolos = list(frecuencia.keys())

Cantidad de caracteres del texto a analizar : 2097953


In [33]:
################################################################################################
# 3. Construcción de la tabla de probabilidades
################################################################################################

# Calcular probabilidades y actualizar el diccionario
probabilidad_total = Decimal(0)
for simbolo in simbolos:
    ocurrencias = frecuencia[simbolo][0]
    probabilidad = Decimal(ocurrencias) / total_chars
    frecuencia[simbolo][1] = probabilidad  # Almacenar la probabilidad
    probabilidad_total += probabilidad

# Preparar los datos para la tabla
tabla_datos = []
for simbolo, datos in frecuencia.items():
    ocurrencias, probabilidad, _ = datos
    bytes_utf8 = len(simbolo.encode('utf-8'))
    imprimible = simbolo if simbolo.isprintable() else repr(simbolo)

    tabla_datos.append([imprimible, ocurrencias, f"{probabilidad:.8f}", bytes_utf8])

# Ordenar los datos por el número de ocurrencias de forma descendente
tabla_datos_ordenada = sorted(tabla_datos, key=lambda x: x[1], reverse=True)

# Encabezados de la tabla
encabezados = ["Símbolo", "Ocurrencias", "Probabilidad", "Bytes en UTF-8"]

# Imprimir la tabla
print("\nTabla de Símbolos y Probabilidades:")
print(tabulate(tabla_datos_ordenada, headers=encabezados, tablefmt="grid"))
print(f"\nSon {len(simbolos)} Caracteres Diferentes")
print(f"La Probabilidad Total es : {probabilidad_total}")
print(f"Total de caracteres en el texto: {total_chars}")

# Calcular el tamaño total de los bytes
tamaño_total_bytes = sum(ocurrencias * len(simbolo.encode('utf-8')) for simbolo, (ocurrencias, _, _) in frecuencia.items())

# Calcular el tamaño promedio en bytes
tamaño_promedio_bytes = Decimal(tamaño_total_bytes) / total_chars

# Imprimir el resultado
print(f"El tamaño promedio en bytes (UTF-8) es: {tamaño_promedio_bytes:.6f} bytes/simbolo en el texto original")


Tabla de Símbolos y Probabilidades:
+-----------+---------------+----------------+------------------+
| Símbolo   |   Ocurrencias |   Probabilidad |   Bytes en UTF-8 |
+===========+===============+================+==================+
|           |        352959 |     0.16824    |                1 |
+-----------+---------------+----------------+------------------+
| e         |        220558 |     0.10513    |                1 |
+-----------+---------------+----------------+------------------+
| a         |        191459 |     0.0912599  |                1 |
+-----------+---------------+----------------+------------------+
| o         |        153000 |     0.0729282  |                1 |
+-----------+---------------+----------------+------------------+
| s         |        122467 |     0.0583745  |                1 |
+-----------+---------------+----------------+------------------+
| n         |        107673 |     0.0513229  |                1 |
+-----------+---------------+----------

In [34]:
################################################################################################
# 4. Cálculo de la entropía
################################################################################################

# En esta parte, se extraen las probabilidades directamente del diccionario 'frecuencia'
# El valor de la probabilidad es el segundo elemento del vector de datos (índice 1)

H = Decimal(0)
# Se itera sobre los valores del diccionario frecuencia
for datos in frecuencia.values():
    probabilidad = datos[1]  # La probabilidad es el segundo elemento del vector
    if probabilidad > 0:
        H = H -  Decimal(probabilidad * Decimal(math.log2(probabilidad)))

print(f"\nEntropía H(X) = {H:.6f} bits/símbolo")


Entropía H(X) = 4.378741 bits/símbolo


In [35]:
################################################################################################
# 5. Construcción del código de Huffman
################################################################################################

def huffman(frecuencias_dict):
    # Se crea la lista inicial para el heap
    # Cada elemento del heap será: [probabilidad, [símbolo, ""]]
    heap = []
    for simbolo, datos in frecuencias_dict.items():
        probabilidad = datos[1]  # La probabilidad es el segundo elemento del vector
        heap.append([probabilidad, [simbolo, ""]])

    heapq.heapify(heap)

    while len(heap) > 1:
        # Se extraen los dos elementos con menor probabilidad del heap
        lo = heapq.heappop(heap)
        hi = heapq.heappop(heap)

        # Se asignan los prefijos '0' y '1' a los códigos
        for pair in lo[1:]:
            pair[1] = '0' + pair[1]
        for pair in hi[1:]:
            pair[1] = '1' + pair[1]

        # Se fusionan los nodos y se insertan de nuevo en el heap
        heapq.heappush(heap, [lo[0] + hi[0]] + lo[1:] + hi[1:])

    # El resultado final es un árbol de un solo nodo.
    # Se extraen los códigos ordenados por longitud y luego alfabéticamente
    return sorted(heapq.heappop(heap)[1:], key=lambda p: (len(p[1]), p[0]))

# Se llama a la función Huffman con el diccionario de frecuencias
codigos = huffman(frecuencia)

# Se crea un diccionario de mapeo para los códigos
codigo_map = {simbolo: codigo for simbolo, codigo in codigos}

# Actualizar el diccionario de frecuencias con los códigos de Huffman
for simbolo, codigo in codigos:
    if simbolo in frecuencia:
        # El tercer elemento del vector se actualiza con el código de Huffman
        frecuencia[simbolo][2] = codigo

In [36]:


from decimal import Decimal

################################################################################################
# 6. Cálculo de longitud promedio y eficiencia
#    Guardar todas las variables que dependen del codigo para no regenerar
#    El algoritmo podria generar una tabla diferente para simbolos con identica probabilidad
################################################################################################

# Inicializar la suma
suma_ponderada = Decimal(0)
acumulador_real = Decimal(0) # se usara para calcular L desde la tabla de ocurrencias y no de posibilidades

# Iterar sobre los valores del diccionario de frecuencias
for datos in frecuencia.values():
    probabilidad = datos[1]  # La probabilidad es el segundo elemento
    ocurrencia = datos[0]    # La ocurrencia del simbolo en la muestra
    codigo = datos[2]        # El código de Huffman es el tercer elemento

    # Obtener la longitud del código de bits
    longitud_codigo = len(codigo)

    # Calcular el valor ponderado (probabilidad * longitud)
    valor_ponderado = probabilidad * longitud_codigo
    acumulador_real = acumulador_real + ocurrencia*longitud_codigo

    # Sumar al total
    suma_ponderada += valor_ponderado

# Asignar el resultado final a la variable L
L = suma_ponderada
L_real = acumulador_real / total_chars

# Calcular la eficiencia
# (Asumiendo que la variable H ya ha sido calculada en el segmento anterior)
eff = H / L
eff_real = H / L_real
print(f"\nLongitud promedio L(C,X) = {L:.5f} bits/símbolo")
print(f"Eficiencia η = {eff * 100:.2f}%")
print("*" * 50)
print(f"\nLongitud real promedio L(C,X) = {L_real:.5f} bits/símbolo")
print(f"Eficiencia real η = {eff_real * 100:.2f}%")


Longitud promedio L(C,X) = 4.41576 bits/símbolo
Eficiencia η = 99.16%
**************************************************

Longitud real promedio L(C,X) = 4.41576 bits/símbolo
Eficiencia real η = 99.16%


In [37]:
############################################################
# Calculo del L_real_calculado
############################################################

# Calcular el total de bits codificados sumando las longitudes de los códigos
# multiplicadas por sus ocurrencias.
bits_reales_codificados = sum(datos[0] * len(datos[2]) for datos in frecuencia.values())

# Calcular el tamaño del archivo en bytes con padding (añadiendo bits hasta el siguiente byte completo)
tamanio_datos_reales = (bits_reales_codificados + 7) // 8

# Calcular L_real
L_real_calculado = (tamanio_datos_reales * 8) / total_chars

print(f"\nBits reales codificados (calculado): {bits_reales_codificados}")
print(f"Tamaño de datos reales (calculado, con padding): {tamanio_datos_reales} bytes")
print(f"L_real (calculado): {L_real_calculado:.4f} bits/símbolo")


Bits reales codificados (calculado): 9264062
Tamaño de datos reales (calculado, con padding): 1158008 bytes
L_real (calculado): 4.4158 bits/símbolo


In [38]:
################################################################################################
# 7. Codificación del texto
################################################################################################

# 1. Codificar el texto a string de bits
cadena_de_bits = ''.join(frecuencia[char][2] for char in texto)

# 2. Calcular y agregar padding
tamanio_de_relleno = (8 - len(cadena_de_bits) % 8) % 8
bits_reales_codificados = len(cadena_de_bits)
cadena_de_bits += '0' * tamanio_de_relleno

# 3. Convertir string de bits a bytes
bits_bytes = bytearray()
for i in range(0, len(cadena_de_bits), 8):
    byte_str = cadena_de_bits[i:i+8]
    byte_val = int(byte_str, 2)
    bits_bytes.append(byte_val)

# 4. CALCULO DE TAMAÑOS Y OVERHEAD
tamanio_overhead = 1 + 2  # 1 byte para padding y 2 para el número de símbolos
for simbolo, codigo in codigos:
    simb_utf8 = simbolo.encode('utf-8')
    bytes_necesarios_codigo = (len(codigo) + 7) // 8
    tamanio_overhead += 1 + len(simb_utf8) + 1 + bytes_necesarios_codigo

tamanio_datos_reales = len(bits_bytes)
tamanio_total_comprimido = tamanio_overhead + tamanio_datos_reales

# 5. Guardar a archivo .huff
ruta_de_archivo = "quijote.huff"
with open(ruta_de_archivo, "wb") as f:
    # Escribir metadata
    f.write(tamanio_de_relleno.to_bytes(1, 'big'))
    f.write(len(codigos).to_bytes(2, 'big'))

    # Escribir diccionario
    for simbolo, codigo in codigos:
        simb_utf8 = simbolo.encode('utf-8')
        f.write(len(simb_utf8).to_bytes(1, 'big'))
        f.write(simb_utf8)
        f.write(len(codigo).to_bytes(1, 'big'))

        # Convertir código binario a bytes
        codigo_int = int(codigo, 2)
        bytes_necesarios = (len(codigo) + 7) // 8
        f.write(codigo_int.to_bytes(bytes_necesarios, 'big'))

    # Escribir datos comprimidos
    f.write(bits_bytes)

In [39]:
################################################################################################
# 8. Decodificación y comparación del texto
################################################################################################

def decodificar_huffman(ruta_de_archivo):
    if not os.path.exists(ruta_de_archivo):
        print(f"Error: El archivo '{ruta_de_archivo}' no existe.")
        return ""

    with open(ruta_de_archivo, "rb") as f:
        tamanio_de_relleno = int.from_bytes(f.read(1), 'big')
        num_simbolos = int.from_bytes(f.read(2), 'big')

        mapa_decodificacion = {}
        for _ in range(num_simbolos):
            len_simb = int.from_bytes(f.read(1), 'big')
            simbolo_bytes = f.read(len_simb)
            simbolo = simbolo_bytes.decode('utf-8')
            len_codigo = int.from_bytes(f.read(1), 'big')
            bytes_necesarios = (len_codigo + 7) // 8
            codigo_int = int.from_bytes(f.read(bytes_necesarios), 'big')
            codigo = bin(codigo_int)[2:].zfill(len_codigo)
            mapa_decodificacion[codigo] = simbolo

        datos_comprimidos = f.read()

    cadena_de_bits = ''.join(bin(byte)[2:].zfill(8) for byte in datos_comprimidos)

    if tamanio_de_relleno > 0:
        cadena_de_bits = cadena_de_bits[:-tamanio_de_relleno]

    texto_decodificado = ""
    codigo_actual = ""
    for bit in cadena_de_bits:
        codigo_actual += bit
        if codigo_actual in mapa_decodificacion:
            texto_decodificado += mapa_decodificacion[codigo_actual]
            codigo_actual = ""

    return texto_decodificado

# Decodificar el archivo y comparar
texto_decodificado = decodificar_huffman(ruta_de_archivo)

print("Verificando la integridad de la decodificación...")

if texto_original == texto_decodificado:
    print("*** El archivo decodificado coincide perfectamente con el original. ***")
else:
    print("\n" + "*" * 70)
    print("¡ERROR! El archivo decodificado NO coincide con el original.")
    print("\n" + "*" * 70)


Verificando la integridad de la decodificación...
*** El archivo decodificado coincide perfectamente con el original. ***


In [41]:
################################################################################################
# 10. Comparación de tamaños de archivo y relación de compresión
################################################################################################

# Obtener los tamaños de los archivos del sistema operativo
tamanio_original = len(texto_original.encode('utf-8'))
tamanio_comprimido = os.path.getsize(ruta_de_archivo)

# Calcular la relación de compresión real
relacion_compresion_real = Decimal(tamanio_original / tamanio_comprimido)

# Imprimir los resultados
print("\n" + "#" * 50)
print("Análisis del Tamaño de Archivo")
print(f"Tamaño del archivo original: {tamanio_original} bytes")
print(f"Tamaño del archivo comprimido: {tamanio_comprimido} bytes")
print(f"Relación de compresión real: {relacion_compresion_real:.2f} veces")
print(f"Reducción de tamaño: {100 * (1 - tamanio_comprimido / tamanio_original):.2f}%")

################################################################################################
# 11. Comparación de métricas teóricas vs. reales
################################################################################################

# Cálculo del tamaño teórico de los datos codificados
# L ya está calculado en el código
# total_chars ya está calculado
bits_teoricos_codificados = L * total_chars

# El tamaño de padding ya ha sido calculado en la sección de codificación
# tamaño_de_relleno ya está calculado
tamanio_teorico_bits_con_padding = bits_teoricos_codificados + tamanio_de_relleno
tamanio_teorico_bytes_datos = (int(tamanio_teorico_bits_con_padding) + 7) // 8

# El overhead del diccionario de códigos ya fue calculado
# tamanio_overhead ya está calculado
tamanio_teorico_total = tamanio_teorico_bytes_datos + tamanio_overhead

# Obtener el tamaño del archivo comprimido real
tamanio_real_total = os.path.getsize(ruta_de_archivo)

# Imprimir los resultados
print("\n" + "#" * 50)
print("Análisis Teórico vs. Real")
print(f"Tamaño teórico total del archivo .huff: {tamanio_teorico_total} bytes")
print(f"Tamaño real del archivo .huff         : {tamanio_real_total} bytes")

print("-" * 50)
diferencia = tamanio_real_total - tamanio_teorico_total
if abs(diferencia) <= 1:
    print("*** La métrica teórica coincide muy bien con el tamaño real. ***")
else:
    print("*" * 70)
    print("La métrica teórica NO COINCIDE exactamente con el tamaño real. !!")
    print(f"Diferencia: {diferencia} bytes")
    print("*" * 70)



##################################################
Análisis del Tamaño de Archivo
Tamaño del archivo original: 2141519 bytes
Tamaño del archivo comprimido: 1158471 bytes
Relación de compresión real: 1.85 veces
Reducción de tamaño: 45.90%

##################################################
Análisis Teórico vs. Real
Tamaño teórico total del archivo .huff: 1158471 bytes
Tamaño real del archivo .huff         : 1158471 bytes
--------------------------------------------------
*** La métrica teórica coincide muy bien con el tamaño real. ***


**Discuta aquí la metodología, decisiones de formato y resultados obenidos**



## Proceso de Compresión y Descompresión con Huffman

Luego de calcular el código de Huffman para cada símbolo, se procede a generar el archivo comprimido. El proceso implica codificar el texto original en una secuencia de bits, agregar relleno (padding) para completar el último byte y, finalmente, guardar los datos codificados junto con la información necesaria para la decodificación.

Más adelante se descomprime el archivo y se comparan los resultados con el archivo original para verificar su integridad.

### 1\. Codificación y Empaquetamiento de Datos

Los siguientes pasos detallan el proceso para transformar el texto original en una secuencia de bytes que se pueda guardar en el archivo comprimido.

  * **Paso 1: Codificar el texto en una cadena de bits**
    Se recorre el texto original, y para cada carácter, se busca su código de Huffman en el diccionario de frecuencias (`frecuencia`). Estos códigos se concatenan para formar una larga cadena de bits.

        cadena_de_bits = ''.join(frecuencia[char][2] for char in texto)

  * **Paso 2: Calcular el relleno (padding)**
    Los datos se deben almacenar en unidades de bytes (múltiplos de 8 bits). El padding es la cantidad de bits de relleno que se añaden al final para que la longitud total de la cadena de bits sea un múltiplo de 8.

        tamanio_de_relleno = (8 - len(cadena_de_bits) % 8) % 8

  * **Paso 3: Agregar el padding**
    Se añaden ceros al final de la cadena de bits para alcanzar la longitud deseada.

        cadena_de_bits += '0' * tamanio_de_relleno

  * **Paso 4: Convertir la cadena de bits a bytes**
    Se divide la cadena de bits en segmentos de 8 bits y cada segmento se convierte en un valor entero que luego se guarda en un `bytearray`.

        bits_bytes = bytearray()
        for i in range(0, len(cadena_de_bits), 8):
            byte_str = cadena_de_bits[i:i+8]
            byte_val = int(byte_str, 2)
            bits_bytes.append(byte_val)

### 2\. Estructura del Archivo `.huff`

El archivo comprimido no solo contiene los datos codificados; también debe incluir un **encabezado** que permita al decodificador reconstruir la tabla de códigos de Huffman. Esta información se denomina **overhead**.

La estructura final del archivo `.huff` es la siguiente:

1.  **Tamaño del padding** (1 byte): La cantidad de bits de relleno. Este valor se usa durante la decodificación para eliminar el padding.
2.  **Número de símbolos únicos** (2 bytes): La cantidad de símbolos en la tabla de Huffman.
3.  **Diccionario de Huffman**: Una serie de entradas, una por cada símbolo. Cada entrada contiene:
      * **Longitud del símbolo en bytes** (1 byte): Esto permite manejar caracteres Unicode de uno o más bytes.
      * **Símbolo** (en codificación UTF-8): El carácter real, como `á`, `ñ`, o `     `.
      * **Longitud del código** (1 byte): La longitud del código de Huffman en bits.
      * **Código de Huffman** (en bytes): El código de bits empaquetado como bytes.
4.  **Datos comprimidos**: La cadena de bits del texto original, ahora convertida en bytes.


### 3\. Generación del Archivo `.huff`

El archivo se genera escribiendo los datos en el orden de la estructura definida.

    # Guardar a archivo .huff
    ruta_de_archivo = "quijote.huff"
    with open(ruta_de_archivo, "wb") as f:
        # 1. Escribir metadata: padding y número de símbolos
        f.write(tamanio_de_relleno.to_bytes(1, 'big'))
        f.write(len(codigos).to_bytes(2, 'big'))
    
        # 2. Escribir diccionario
        for simbolo, codigo in codigos:
            simb_utf8 = simbolo.encode('utf-8')
            f.write(len(simb_utf8).to_bytes(1, 'big'))
            f.write(simb_utf8)
            f.write(len(codigo).to_bytes(1, 'big'))
        
            # Convertir el código binario a bytes y escribirlo
            codigo_int = int(codigo, 2)
            bytes_necesarios = (len(codigo) + 7) // 8
            f.write(codigo_int.to_bytes(bytes_necesarios, 'big'))
        
        # 3. Escribir los datos comprimidos
        f.write(bits_bytes)



### 4\. Decodificación y Verificación

El proceso de decodificación es la operación inversa. El programa lee el archivo `.huff`, reconstruye la tabla de códigos, lee los datos comprimidos y los decodifica bit a bit. Finalmente, el texto recuperado se compara con el original para confirmar que la decodificación se realizó sin errores.

### 5\. Resultados

    ##################################################
    Análisis del Tamaño de Archivo
    Tamaño del archivo original: 2141519 bytes
    Tamaño del archivo comprimido: 1158471 bytes
    Relación de compresión real: 1.85 veces
    Reducción de tamaño: 45.90%

    ##################################################
    Análisis Teórico vs. Real
    Tamaño teórico total del archivo .huff: 1158471 bytes
    Tamaño real del archivo .huff         : 1158471 bytes
    --------------------------------------------------
    *** La métrica teórica coincide muy bien con el tamaño real. ***